# XGBoost ハイパーパラメータチューニング with Snowflake Experiment Tracking

## 概要
Feature Store (SALES_FORECAST_FV v2) から特徴量を取得し、
複数のXGBoostパラメータセットを **Snowflake Experiment Tracking** で比較評価します。
ベストモデルを **Model Registry** に登録し、**SPCS** にデプロイします。

### 実装ステップ
1. **データ準備**: Feature Store から特徴量取得 → Train/Test 分割
2. **ハイパーパラメータ探索**: 8パターンのXGBoostパラメータを実験トラッキングで記録
3. **結果比較**: RMSE / MAPE で全Runを比較・可視化
4. **ベストモデル登録**: Model Registry に登録（SQL推論対応）
5. **推論テスト**: 登録モデルで推論検証
6. **SPCSデプロイ**: リアルタイム推論サービスとして公開

### 前提条件
- `FOODEX_DEMO.SALES_ML` スキーマが作成済み
- Feature View `SALES_FORECAST_FV` (v2) が登録済み
- Compute Pool `SALES_FORECAST_POOL` が作成済み（SPCSデプロイ時に使用）

## Step 1: 環境セットアップ & Feature Store からデータ取得

In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F
from snowflake.ml.feature_store import FeatureStore, CreationMode
from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.modeling.xgboost import XGBRegressor
from snowflake.ml.modeling.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import numpy as np
import pandas as pd
import time
from datetime import timedelta

session = get_active_session()
session.sql("USE DATABASE FOODEX_DEMO").collect()
session.sql("USE SCHEMA SALES_ML").collect()
session.sql("USE WAREHOUSE COMPUTE_WH").collect()

print("Database:  FOODEX_DEMO")
print("Schema:    SALES_ML")
print("Warehouse: COMPUTE_WH")

In [ ]:
fs = FeatureStore(
    session=session,
    database="FOODEX_DEMO",
    name="SALES_ML",
    default_warehouse="COMPUTE_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

sales_fv = fs.get_feature_view("SALES_FORECAST_FV", "v2")
print("Feature View loaded: FOODEX_DEMO.SALES_ML.SALES_FORECAST_FV (v2)")

## Step 2: Train/Test 分割 & エンコーディング

Feature Store から特徴量を取得し、時系列ベースで分割します。
直近30日間をテストデータとして使用します。

In [ ]:
features_df = fs.read_feature_view(sales_fv)

max_date = features_df.select(F.max('SALES_DATE')).collect()[0][0]
test_start_date = max_date - timedelta(days=29)

train_df = features_df.filter(F.col('SALES_DATE') < test_start_date)
test_df = features_df.filter(F.col('SALES_DATE') >= test_start_date)

print(f"データ最終日:   {max_date}")
print(f"テスト開始日:   {test_start_date}")
print(f"Train records:  {train_df.count()}")
print(f"Test records:   {test_df.count()}")

In [ ]:
label_encoder = LabelEncoder(
    input_cols=["CATEGORY_MEDIUM"],
    output_cols=["CATEGORY_ENCODED"]
)
label_encoder.fit(train_df)
train_encoded = label_encoder.transform(train_df)
test_encoded = label_encoder.transform(test_df)

feature_cols = ['LAG_1', 'LAG_2', 'LAG_3', 'LAG_7', 'LAG_14', 'MA_7',
                'DAY_OF_WEEK', 'IS_WEEKEND', 'MONTH_NUM', 'QUARTER_NUM',
                'TXN_COUNT', 'TOTAL_QTY', 'CATEGORY_ENCODED']
target_col = 'DAILY_SALES'

print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Target: {target_col}")

In [ ]:
train_pd = train_encoded.to_pandas()
test_pd = test_encoded.to_pandas()

print(f"Pandas DataFrames prepared:")
print(f"  train_pd: {train_pd.shape}")
print(f"  test_pd:  {test_pd.shape}")

## Step 3: ハイパーパラメータ探索 with Experiment Tracking

**Snowflake Experiment Tracking** はMLflowベースの実験管理機能です。
各パラメータセットごとに **Run** を作成し、以下を自動記録します:

| 記録項目 | 内容 |
|---------|------|
| Parameters | n_estimators, max_depth, learning_rate, subsample, colsample_bytree |
| Metrics | RMSE, MAPE, training_time_sec |

### パラメータグリッド（8パターン）:
- `n_estimators`: 100, 200, 300
- `max_depth`: 4, 6, 8
- `learning_rate`: 0.05, 0.1
- `subsample`: 0.7, 0.8, 0.9

In [ ]:
exp = ExperimentTracking(session=session)
exp.set_experiment("XGBOOST_HYPERPARAMETER_TUNING")

print("Experiment: XGBOOST_HYPERPARAMETER_TUNING")
print("Schema:     FOODEX_DEMO.SALES_ML")

In [ ]:
param_grid = [
    {"n_estimators": 100, "max_depth": 4, "learning_rate": 0.1, "subsample": 0.8},
    {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.1, "subsample": 0.8},
    {"n_estimators": 100, "max_depth": 6, "learning_rate": 0.1, "subsample": 0.8},
    {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.1, "subsample": 0.8},
    {"n_estimators": 100, "max_depth": 8, "learning_rate": 0.05, "subsample": 0.8},
    {"n_estimators": 200, "max_depth": 8, "learning_rate": 0.05, "subsample": 0.8},
    {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.05, "subsample": 0.9},
    {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.1, "subsample": 0.7, "colsample_bytree": 0.7},
]

print(f"パラメータセット数: {len(param_grid)}")
for i, p in enumerate(param_grid):
    print(f"  [{i+1}] {p}")

In [ ]:
from xgboost import XGBRegressor as XGBRegressorNative

run_timestamp = int(time.time())
results = []

print("=" * 70)
print("  Hyperparameter Search Start")
print("=" * 70)

for i, params in enumerate(param_grid):
    lr_str = str(params['learning_rate']).replace('.', '_')
    run_name = f"xgb_n{params['n_estimators']}_d{params['max_depth']}_lr{lr_str}_{run_timestamp}_{i}"
    print(f"\n[{i+1}/{len(param_grid)}] {run_name}")

    try:
        exp.end_run()
    except:
        pass

    exp.start_run(run_name)

    full_params = {
        "n_estimators": params.get("n_estimators", 100),
        "max_depth": params.get("max_depth", 6),
        "learning_rate": params.get("learning_rate", 0.1),
        "subsample": params.get("subsample", 0.8),
        "colsample_bytree": params.get("colsample_bytree", 0.8),
        "random_state": 42
    }

    exp.log_params(full_params)
    exp.log_param("feature_count", len(feature_cols))

    model = XGBRegressorNative(**full_params)

    start_time = time.time()
    model.fit(train_pd[feature_cols], train_pd[target_col])
    training_time = time.time() - start_time

    predictions = model.predict(test_pd[feature_cols])

    rmse = np.sqrt(mean_squared_error(test_pd[target_col], predictions))
    mape = mean_absolute_percentage_error(test_pd[target_col], predictions) * 100

    exp.log_metric("RMSE", rmse)
    exp.log_metric("MAPE", mape)
    exp.log_metric("training_time_sec", training_time)

    exp.end_run()

    results.append({
        "run_name": run_name,
        "n_estimators": full_params["n_estimators"],
        "max_depth": full_params["max_depth"],
        "learning_rate": full_params["learning_rate"],
        "subsample": full_params["subsample"],
        "colsample_bytree": full_params["colsample_bytree"],
        "RMSE": rmse,
        "MAPE": mape,
        "training_time": training_time
    })

    print(f"  RMSE: {rmse:,.2f}, MAPE: {mape:.2f}%, Time: {training_time:.1f}s")

print("\n" + "=" * 70)
print("  Hyperparameter Search Complete")
print("=" * 70)

## Step 4: 結果比較 & 可視化

全Runの結果をDataFrameに集約し、RMSE/MAPEで比較します。
Snowsight UI でも確認可能: `AI&ML → Experiments → XGBOOST_HYPERPARAMETER_TUNING`

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('RMSE')

print("=== 全Run結果（RMSE昇順） ===")
print(results_df[['n_estimators', 'max_depth', 'learning_rate', 'subsample',
                   'colsample_bytree', 'RMSE', 'MAPE', 'training_time']].to_string(index=False))

best_run = results_df.iloc[0]
print(f"\n>>> ベストモデル: RMSE={best_run['RMSE']:,.2f}, MAPE={best_run['MAPE']:.2f}%")
print(f"    n_estimators={int(best_run['n_estimators'])}, max_depth={int(best_run['max_depth'])}, "
      f"lr={best_run['learning_rate']}, subsample={best_run['subsample']}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

labels = [f"N{int(r['n_estimators'])}_D{int(r['max_depth'])}_LR{r['learning_rate']}" for _, r in results_df.iterrows()]

ax1 = axes[0]
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(results_df))]
ax1.barh(labels, results_df['RMSE'], color=colors)
ax1.set_xlabel('RMSE (円)')
ax1.set_title('RMSE 比較（低いほど良い）')
ax1.grid(axis='x', alpha=0.3)
ax1.invert_yaxis()

ax2 = axes[1]
results_mape_sorted = results_df.sort_values('MAPE')
labels_mape = [f"N{int(r['n_estimators'])}_D{int(r['max_depth'])}_LR{r['learning_rate']}" for _, r in results_mape_sorted.iterrows()]
colors_mape = ['#2ecc71' if i == 0 else '#e74c3c' for i in range(len(results_mape_sorted))]
ax2.barh(labels_mape, results_mape_sorted['MAPE'], color=colors_mape)
ax2.set_xlabel('MAPE (%)')
ax2.set_title('MAPE 比較（低いほど良い）')
ax2.grid(axis='x', alpha=0.3)
ax2.invert_yaxis()

ax3 = axes[2]
scatter = ax3.scatter(results_df['RMSE'], results_df['MAPE'],
                      c=results_df['training_time'], cmap='YlOrRd',
                      s=100, edgecolors='black', linewidths=0.5)
ax3.set_xlabel('RMSE (円)')
ax3.set_ylabel('MAPE (%)')
ax3.set_title('RMSE vs MAPE（色=学習時間）')
ax3.grid(alpha=0.3)
plt.colorbar(scatter, ax=ax3, label='Training Time (s)')

best_idx = results_df.index[0]
ax3.annotate('BEST', xy=(results_df.loc[best_idx, 'RMSE'], results_df.loc[best_idx, 'MAPE']),
             fontsize=10, fontweight='bold', color='green',
             xytext=(10, 10), textcoords='offset points',
             arrowprops=dict(arrowstyle='->', color='green'))

plt.tight_layout()
plt.show()

## Step 5: ベストモデルを Model Registry に登録

実験トラッキングで最良と判断されたモデルを
**Snowflake Model Registry** に登録し、SQL推論を可能にします。

### 登録後のメリット:
- SQLクエリから直接予測を実行可能
- バージョン管理（v1, v2...）
- 本番環境への安全なデプロイ

In [ ]:
best_params = {
    "n_estimators": int(best_run['n_estimators']),
    "max_depth": int(best_run['max_depth']),
    "learning_rate": best_run['learning_rate'],
    "subsample": best_run['subsample'],
    "colsample_bytree": best_run['colsample_bytree'],
    "random_state": 42
}

print("Best Model Parameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

In [ ]:
from snowflake.ml.modeling.xgboost import XGBRegressor as SnowparkXGBRegressor

best_model = SnowparkXGBRegressor(
    input_cols=feature_cols,
    label_cols=[target_col],
    output_cols=["PREDICTED_SALES"],
    **best_params
)

print("Training best model with Snowpark ML...")
best_model.fit(train_encoded)
print("Training completed.")

In [ ]:
best_model_native = XGBRegressorNative(**best_params)
best_model_native.fit(train_pd[feature_cols], train_pd[target_col])

predictions = best_model_native.predict(test_pd[feature_cols])

best_rmse = np.sqrt(mean_squared_error(test_pd[target_col], predictions))
best_mape = mean_absolute_percentage_error(test_pd[target_col], predictions) * 100

print(f"Best Model Metrics:")
print(f"  RMSE: {best_rmse:,.2f} 円")
print(f"  MAPE: {best_mape:.2f} %")

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.registry.model_version import TargetPlatform

registry = Registry(
    session=session,
    database_name="FOODEX_DEMO",
    schema_name="SALES_ML"
)

print("Model Registry initialized: FOODEX_DEMO.SALES_ML")

In [ ]:
sample_input = train_encoded.select(feature_cols).limit(100)

model_version = registry.log_model(
    model=best_model,
    model_name="SALES_FORECAST_MODEL",
    version_name="v2",
    target_platforms=[TargetPlatform.WAREHOUSE],
    sample_input_data=sample_input,
    metrics={
        "RMSE": best_rmse,
        "MAPE": best_mape,
        "n_estimators": best_params["n_estimators"],
        "max_depth": best_params["max_depth"],
        "learning_rate": best_params["learning_rate"]
    },
    comment="Best model from hyperparameter tuning experiment"
)

print(f"\nModel registered successfully!")
print(f"  Name:    FOODEX_DEMO.SALES_ML.{model_version.model_name}")
print(f"  Version: {model_version.version_name}")
print(f"  RMSE:    {best_rmse:,.2f}")
print(f"  MAPE:    {best_mape:.2f}%")
print(f"  SQL inference enabled (target_platforms=WAREHOUSE)")

## Step 6: 推論テスト

登録したモデルでSQL推論をテストします。
`model_version.run()` でSnowpark DataFrameを渡して推論を実行できます。

In [ ]:
print("\nRegistered Models:")
for model in registry.models():
    print(f"  - {model.name}")
    for version in model.versions():
        print(f"      Version: {version.version_name}")
        try:
            print(f"      RMSE: {version.get_metric('RMSE')}, MAPE: {version.get_metric('MAPE')}")
        except:
            pass

In [ ]:
print("Testing inference with registered model...")
test_sample = test_pd[feature_cols].head(10)
predictions_test = best_model_native.predict(test_sample)

print("\nSample Predictions:")
print(f"{'':>4} {'Predicted':>12} {'Actual':>12} {'Error%':>10}")
print("-" * 42)
for i, (pred, actual) in enumerate(zip(predictions_test, test_pd[target_col].head(10))):
    error_pct = abs(pred - actual) / actual * 100 if actual != 0 else 0
    print(f"  [{i+1:>2}] {pred:>10,.0f}  {actual:>10,.0f}  {error_pct:>8.1f}%")

In [ ]:
print("Model Registry SQL Inference Test:")
sql_test_result = model_version.run(test_encoded.select(feature_cols).limit(20))
sql_test_result.show()

## Step 7: SPCSでモデルをデプロイ（リアルタイム推論）

**SPCS (Snowpark Container Services)** にモデルをデプロイすると:
- REST APIエンドポイントとして公開
- リアルタイム推論が可能
- オートスケーリング対応

### 使用するCompute Pool:
- `SALES_FORECAST_POOL` (CPU_X64_S, 1-3 nodes)

In [ ]:
model = registry.get_model("SALES_FORECAST_MODEL")
mv = model.version("V2")

print(f"Model:   {model.name}")
print(f"Version: V2")
print(f"\nDeploying to SPCS...")

In [ ]:
service = mv.create_service(
    service_name="SALES_FORECAST_SERVICE",
    service_compute_pool="SALES_FORECAST_POOL",
    image_build_compute_pool="SALES_FORECAST_POOL",
    num_workers=1,
    max_batch_rows=100
)

print(f"Service created: {service.name}")
print(f"Status: {service.status}")

In [ ]:
print("Waiting for service to be ready...")
for i in range(30):
    status = session.sql("DESCRIBE SERVICE FOODEX_DEMO.SALES_ML.SALES_FORECAST_SERVICE").collect()
    current_status = status[0]['status'] if status else 'UNKNOWN'
    print(f"  [{i+1}/30] Status: {current_status}")
    if current_status == 'READY':
        print("\nService is READY!")
        break
    time.sleep(10)
else:
    print("\nService is still starting. Check status later.")

## まとめ

### 実験結果
8パターンのXGBoostハイパーパラメータを比較評価し、
ベストモデルを **FOODEX_DEMO.SALES_ML** に登録しました。

### 作成・更新したオブジェクト

| オブジェクト | 場所 | 説明 |
|------------|------|------|
| Experiment | FOODEX_DEMO.SALES_ML | XGBOOST_HYPERPARAMETER_TUNING (8 Runs) |
| Model | FOODEX_DEMO.SALES_ML | SALES_FORECAST_MODEL (v2) |
| Service | FOODEX_DEMO.SALES_ML | SALES_FORECAST_SERVICE (SPCS) |

### 確認方法
- **実験結果**: Snowsight → `AI&ML → Experiments → XGBOOST_HYPERPARAMETER_TUNING`
- **登録モデル**: Snowsight → `AI&ML → Models → SALES_FORECAST_MODEL`
- **SPCSサービス**: Snowsight → `AI&ML → Models → SALES_FORECAST_MODEL → Services`

In [ ]:
print("=" * 60)
print("  Experiment Tracking & Model Registration Complete")
print("=" * 60)
print(f"\n[実験]")
print(f"  Experiment:     XGBOOST_HYPERPARAMETER_TUNING")
print(f"  Total Runs:     {len(results)}")
print(f"\n[ベストモデル]")
print(f"  RMSE:           {best_rmse:,.2f} 円")
print(f"  MAPE:           {best_mape:.2f} %")
print(f"  n_estimators:   {best_params['n_estimators']}")
print(f"  max_depth:      {best_params['max_depth']}")
print(f"  learning_rate:  {best_params['learning_rate']}")
print(f"\n[登録先]")
print(f"  Model:   FOODEX_DEMO.SALES_ML.SALES_FORECAST_MODEL (v2)")
print(f"  Service: FOODEX_DEMO.SALES_ML.SALES_FORECAST_SERVICE")